DATA INGESTION PIPELINE- From data ingestion to vector db

In [1]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

c:\Users\JAYA AGRWAL\Desktop\genai codes\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def process_all_pdf(pdf_directory):
    #Aim is to process all the pdf's in the directory
    all_documents=[]
    pdf_dir=Path(pdf_directory)

    #Find all the pdf files only
    pdf_files=list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files in the directory.")

    for i in pdf_files:
        print(f"Processing file: {i.name}")
        try:
            loader=PyPDFLoader(str(i))
            documents=loader.load()
            for doc in documents:
                doc.metadata["source_file"]=i.name
                doc.metadata["file_type"]="pdf"
            all_documents.extend(documents)
            print(f"Successfully processed {i.name} with PyPDFLoader.")
            print(f"Loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error: {e}")

    print(f"Total pages loaded: {len(all_documents)}")
    return all_documents

all_docs=process_all_pdf("../data")

Found 3 PDF files in the directory.
Processing file: 6.POS.pdf
Successfully processed 6.POS.pdf with PyPDFLoader.
Loaded 11 pages
Processing file: 7.Vectorization.pdf
Successfully processed 7.Vectorization.pdf with PyPDFLoader.
Loaded 10 pages
Processing file: 8.LanguageModels.pdf
Successfully processed 8.LanguageModels.pdf with PyPDFLoader.
Loaded 55 pages
Total pages loaded: 76


In [3]:
all_docs

[Document(metadata={'producer': 'Adobe Scan for Android 23.12.15-google-dynamic', 'creator': 'Adobe Scan for Android 23.12.15-google-dynamic', 'creationdate': '', 'source': '..\\data\\pdf\\6.POS.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'source_file': '6.POS.pdf', 'file_type': 'pdf'}, page_content='2 \ntu Puoblem tags \ni e aing det is the Dnl in e amhigueus \nStochaatic Pos tagg \nPaob. \nNO \nDOMS \nMeaauumet enelntesd mat quety \nDate \nWa wy \nmay yield inadmisi ble den,-9 \nThe b.est tag toa gen Lad is dellmimol y e pobabi lty tat \nw hekune So, Loil dooe \nPage No. \nioleity Pos tag Aulineed to ioliity weros tag \nsCCUing Iwauis tainy epus \npAewius tags \nNeuns appeeiy wiu tag Nen.. e paob. aty'),
 Document(metadata={'producer': 'Adobe Scan for Android 23.12.15-google-dynamic', 'creator': 'Adobe Scan for Android 23.12.15-google-dynamic', 'creationdate': '', 'source': '..\\data\\pdf\\6.POS.pdf', 'total_pages': 11, 'page': 1, 'page_label': '2', 'source_file': '6.POS.p

In [4]:
#Text splitting or basically splitting the documents into chunks
def split_documents(documents,chunk_size=1000,chunk_overlap=200):   
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
        )
    
    split_docs=text_splitter.split_documents(documents)#built in function split_documents
    print(f"split {len(documents)} documents into {len(split_docs)} chunks.")

    if split_docs:
      print(f"Example chunk content:\n{split_docs[0].page_content[:500]}...") 
      print(f"Metadata of the chunk:\n{split_docs[0].metadata}")

    return split_docs    
    

In [5]:
chunks=split_documents(all_docs)
chunks

split 76 documents into 66 chunks.
Example chunk content:
2 
tu Puoblem tags 
i e aing det is the Dnl in e amhigueus 
Stochaatic Pos tagg 
Paob. 
NO 
DOMS 
Meaauumet enelntesd mat quety 
Date 
Wa wy 
may yield inadmisi ble den,-9 
The b.est tag toa gen Lad is dellmimol y e pobabi lty tat 
w hekune So, Loil dooe 
Page No. 
ioleity Pos tag Aulineed to ioliity weros tag 
sCCUing Iwauis tainy epus 
pAewius tags 
Neuns appeeiy wiu tag Nen.. e paob. aty...
Metadata of the chunk:
{'producer': 'Adobe Scan for Android 23.12.15-google-dynamic', 'creator': 'Adobe Scan for Android 23.12.15-google-dynamic', 'creationdate': '', 'source': '..\\data\\pdf\\6.POS.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'source_file': '6.POS.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Adobe Scan for Android 23.12.15-google-dynamic', 'creator': 'Adobe Scan for Android 23.12.15-google-dynamic', 'creationdate': '', 'source': '..\\data\\pdf\\6.POS.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'source_file': '6.POS.pdf', 'file_type': 'pdf'}, page_content='2 \ntu Puoblem tags \ni e aing det is the Dnl in e amhigueus \nStochaatic Pos tagg \nPaob. \nNO \nDOMS \nMeaauumet enelntesd mat quety \nDate \nWa wy \nmay yield inadmisi ble den,-9 \nThe b.est tag toa gen Lad is dellmimol y e pobabi lty tat \nw hekune So, Loil dooe \nPage No. \nioleity Pos tag Aulineed to ioliity weros tag \nsCCUing Iwauis tainy epus \npAewius tags \nNeuns appeeiy wiu tag Nen.. e paob. aty'),
 Document(metadata={'producer': 'Adobe Scan for Android 23.12.15-google-dynamic', 'creator': 'Adobe Scan for Android 23.12.15-google-dynamic', 'creationdate': '', 'source': '..\\data\\pdf\\6.POS.pdf', 'total_pages': 11, 'page': 1, 'page_label': '2', 'source_file': '6.POS.p

In [6]:
##embedding and vector store db
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Tuple, Any
from sklearn.metrics.pairwise import cosine_similarity


In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1401.10it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\JAYA AGRWAL\AppData\Local\Temp\ipykernel_25752\2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [8]:
import os
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0
